In [0]:
from pyspark.sql.functions import *



In [0]:
SOURCE_TABLE = "workspace.silver.historical_hourly"

TARGET_TABLE = "workspace.gold.monthly_weather_summary"

silver_df = spark.table(SOURCE_TABLE)

In [0]:
monthly_summary_df = (
    silver_df
    .groupBy(
        "city_name",
        "year",
        "month"
    )
    .agg(
        avg("temperature").alias("avg_temperature"),
        max("temperature").alias("max_temperature"),
        min("temperature").alias("min_temperature"),

        avg("humidity").alias("avg_humidity"),

        sum("precipitation_lwe").alias("total_precipitation"),

        avg("wind_speed").alias("avg_wind_speed"),

        avg("cloud_cover_total").alias("avg_cloud_cover"),

        count("*").alias("total_observations")
    )
)

In [0]:
monthly_summary_df = (
    monthly_summary_df
    .withColumn("avg_temperature",round("avg_temperature",2))
    .withColumn("avg_humidity",round("avg_humidity",2))
    .withColumn("avg_wind_speed",round("avg_wind_speed",2))
    .withColumn("avg_cloud_cover",round("avg_cloud_cover",2))
    .withColumn("total_precipitation",round("total_precipitation",2))
)

In [0]:
(
    monthly_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)